# 1. Configuración del Entorno y Carga de Componentes

## 1.1. Importación de Librerías

Importamos las librerías necesarias para el análisis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import scanpy as sc
import joblib

# Configuramos el estilo de las visualizaciones
sns.set_theme(style="whitegrid")
print("Librerías importadas correctamente.")

## 1.2. Definición de Rutas

Definimos las rutas a los datos de entrada y a las carpetas de salida.

In [ ]:
DATA_PROCESSED_PATH = '../data/processed/'
MODELS_PATH = '../outputs/models/'
FIGURES_PATH = '../outputs/figures/'

# Nombres de los ficheros de entrada
BULK_COUNTS_FILENAME = 'TCGA-LUAD_counts_for_deconvolution.parquet'
BULK_CLINICAL_FILENAME = 'TCGA-LUAD_clinical_for_deconvolution.parquet'
REF_SC_FILENAME = 'lung_cancer_processed_for_modeling.h5ad'
MODEL_FILENAME = 'mlp_marker_genes.joblib'

os.makedirs(os.path.join(FIGURES_PATH, 'deconvolution'), exist_ok=True)

print("Rutas definidas.")

## 1.3. Carga de Datos de TCGA

Cargamos los datos de expresión y clínicos de la cohorte de TCGA-LUAD, que fueron procesados en el notebook anterior.

In [ ]:
print("Cargando datos de TCGA (bulk RNA-seq)...")
bulk_counts_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_COUNTS_FILENAME))
bulk_clinical_df = pd.read_parquet(os.path.join(DATA_PROCESSED_PATH, BULK_CLINICAL_FILENAME))

print("Datos de TCGA cargados:")
print(f"  - Matriz de conteos: {bulk_counts_df.shape[0]} muestras x {bulk_counts_df.shape[1]} genes")
print(f"  - Datos clínicos: {bulk_clinical_df.shape[0]} muestras x {bulk_clinical_df.shape[1]} variables")


## 1.4. Carga de Datos de Referencia

Cargamos el objeto AnnData de scRNA-seq que contiene los perfiles de expresión de referencia para cada tipo celular.


In [ ]:
print("\nCargando datos de referencia (scRNA-seq)...")
adata_ref = sc.read_h5ad(os.path.join(DATA_PROCESSED_PATH, REF_SC_FILENAME))

print("Datos de referencia cargados:")
print(adata_ref)

## 1.5. Carga del Modelo MLP seleccionado

Cargamos nuestro modelo MLP final, que fue entrenado y validado en el notebook anterior.

In [ ]:
print("\nCargando el modelo MLP entrenado...")
model_path = os.path.join(MODELS_PATH, MODEL_FILENAME)

# El modelo MLP se guardó como un diccionario con el modelo y el LabelEncoder
loaded_model_data = joblib.load(model_path)

# Verificamos si es un diccionario o el modelo directamente
if isinstance(loaded_model_data, dict):
    model = loaded_model_data['model']
    label_encoder = loaded_model_data.get('label_encoder', None)
else:
    model = loaded_model_data
    label_encoder = None

print("Modelo MLP cargado exitosamente.")
if label_encoder:
    print("LabelEncoder también cargado.")
    print("Clases del modelo:", label_encoder.classes_)
else:
    print("Clases del modelo:", model.classes_)

# Nos aseguramos de que los genes en los datos de bulk y de referencia son consistentes.
common_genes = list(set(bulk_counts_df.columns) & set(adata_ref.var.index))
print(f"\n[VERIFICACIÓN] Se encontraron {len(common_genes)} genes en común entre los datos de bulk y de referencia.")

try:
    assert len(common_genes) > 20000, "Hay pocos genes en común. Revisar la anotación de genes (Ensembl IDs)."
    print("[OK] El solapamiento de genes es alto")
except AssertionError as e:
    print(f"[ERROR] {e}")


Próximo, hacer la matriz de firmas y la deconvolución

# 2. Creación de la Matriz de Firmas Genéticas

## 2.1. Selección de los Genes Marcadores

El primer paso es definir qué genes formarán la base de nuestra firma. Para mantener la consistencia con nuestro modelo MLP optimizado, utilizaremos exactamente la misma lista de genes marcadores que se generó en el Notebook 2.

In [ ]:
print("--- Re-calculando la lista de genes marcadores desde los datos de referencia ---")

# Usamos rank_genes_groups en el objeto de referencia para obtener los marcadores
sc.tl.rank_genes_groups(adata_ref, groupby='cell_type', method='t-test', use_raw=False)

# Extraemos el DataFrame de marcadores
marker_genes_df = pd.DataFrame(adata_ref.uns['rank_genes_groups']['names'])

# Definimos cuántos genes por tipo celular
top_n_genes = 25 # Mismo valor que en el notebook anterior

# Creamos la lista final de Ensembl IDs
marker_genes_list = []
for col in marker_genes_df.columns:
    marker_genes_list.extend(marker_genes_df[col].head(top_n_genes))

# Eliminamos duplicados para tener la lista única de features
marker_genes_list = sorted(list(set(marker_genes_list)))

print(f"Se ha generado una lista de {len(marker_genes_list)} genes marcadores únicos.")

## 2.2. Cálculo de la Expresión Promedio por Tipo Celular

A continuación, calculamos la expresión promedio de cada gen marcador en cada tipo celular de nuestro dataset de referencia. Estos perfiles promedio formarán las "firmas" de cada población celular.


In [ ]:
print("\n--- Calculando perfiles de expresión promedio ---")

# Creamos un DataFrame a partir de la matriz de expresión log-normalizada de adata_ref
# Usamos .var_names para los nombres de las columnas (genes)
ref_expression_df = pd.DataFrame(
    adata_ref.X.toarray(),
    index=adata_ref.obs.index,
    columns=adata_ref.var.index
)

# Añadimos la columna 'cell_type' para poder agrupar
ref_expression_df['cell_type'] = adata_ref.obs['cell_type'].values

# Agrupamos por tipo celular y calculamos la media de la expresión de cada gen
signature_matrix = ref_expression_df.groupby('cell_type').mean()

# La matriz resultante tiene tipos celulares como filas y TODOS los genes como columnas.
# La transponemos y filtramos para quedarnos solo con nuestros genes marcadores.
signature_matrix = signature_matrix.T
signature_matrix = signature_matrix.loc[marker_genes_list]

print("Matriz de firmas genéticas creada con éxito.")
print(f"Dimensiones de la matriz de firmas: {signature_matrix.shape[0]} genes x {signature_matrix.shape[1]} tipos celulares")
display(signature_matrix.head())

## 2.3. Visualización de la Matriz de Firmas

Para verificar visualmente la calidad de nuestra matriz de firmas, la representamos como un heatmap. Esperamos ver patrones claros donde los genes marcadores muestran una alta expresión específica en su tipo celular correspondiente.

In [ ]:
print("\n--- Visualizando la matriz de firmas ---")

# Usamos un clustering jerárquico para agrupar genes y tipos celulares similares
# Esto a menudo revela bloques de co-expresión.
plt.figure(figsize=(12, 18))
sns.clustermap(
    signature_matrix,
    cmap='viridis',       
    standard_scale=0,     # Escalamos por gen (filas) para resaltar patrones relativos
    dendrogram_ratio=0.1
)
plt.suptitle('Heatmap Clusterizado de la Matriz de Firmas Genéticas', y=1.02)
plt.show()

# 3. Deconvolución de las Muestras de Bulk RNA-seq

## 3.1. Preparación de los Datos

Antes de ejecutar el algoritmo, debemos asegurar que nuestra matriz de conteos de bulk y nuestra matriz de firmas estén perfectamente alineadas, usando exactamente el mismo conjunto de genes marcadores y en el mismo orden. También es crucial que los datos estén en la misma escala (espacio lineal, no logarítmico) para el modelo de regresión.

In [ ]:
from sklearn.svm import NuSVR
from tqdm.notebook import tqdm

print("--- Preparando los datos para la deconvolución ---")

# 1. Alinear los genes
# Nos aseguramos de que la matriz de bulk solo contenga los genes de nuestra firma
common_genes_deconv = list(set(signature_matrix.index) & set(bulk_counts_df.columns))
print(f"Alineando sobre {len(common_genes_deconv)} genes marcadores comunes.")

# Filtramos y reordenamos ambas matrices para que coincidan
signature_matrix_aligned = signature_matrix.loc[common_genes_deconv]
bulk_counts_aligned = bulk_counts_df[common_genes_deconv]

# 2. Verificar la escala de los datos
# Los modelos de regresión para deconvolución suelen funcionar mejor en espacio lineal.
# Nuestra matriz de firmas se calculó sobre datos log-normalizados.
# Vamos a revertir la transformación logarítmica (exp(x) - 1) para volver a una escala
# más parecida a los conteos.
# Esta es una aproximación, ya que los datos de bulk también deberían ser
# normalizados (ej. TPM) pero no logaritmizados. Para este enfoque, usaremos
# los conteos crudos de bulk y la firma "des-logaritmizada".

# Revertimos el log2(x+1) de la matriz de firmas.
# Primero, volvemos a la base natural: log(x+1) = log2(x+1) * log(2)
# Luego, exp(log(x+1)) - 1 = x
signature_matrix_linear = np.expm1(signature_matrix_aligned * np.log(2))

# Usaremos los conteos crudos de bulk
bulk_linear_df = bulk_counts_aligned

## 3.2. Ejecución de la Deconvolución Basada en nu-SVR

Ahora, iteramos sobre cada una de las 539 muestras tumorales. Para cada una, ajustamos un modelo de regresión de soporte vectorial (nu-SVR) para estimar las proporciones de los 10 tipos celulares de nuestra firma.

In [ ]:
print("\n--- Iniciando el proceso de deconvolución ---")

# Preparamos los datos para sklearn
X_signature = signature_matrix_linear.values # (tipos_celulares x genes)
cell_types = signature_matrix_linear.columns

# Lista para guardar los resultados
all_proportions = []

# Usamos tqdm para visualizar el progreso del bucle
for sample_id in tqdm(bulk_linear_df.index, desc="Deconvolucionando muestras"):
    # Obtenemos el perfil de expresión de la muestra actual
    y_bulk_sample = bulk_linear_df.loc[sample_id].values
    
    # Definimos y entrenamos el modelo de regresión
    # nu=0.5 es un valor estándar. kernel='linear' porque asumimos una mezcla aditiva.
    model_svr = NuSVR(kernel='linear', nu=0.5)
    model_svr.fit(X_signature, y_bulk_sample)
    
    # Obtenemos los coeficientes del modelo. Estos son nuestras "proporciones" crudas.
    raw_proportions = model_svr.coef_.copy()
    
    # Post-procesamiento de los coeficientes:
    # 1. Forzar a que no sean negativos (biológicamente no tiene sentido una proporción negativa)
    raw_proportions[raw_proportions < 0] = 0
    
    # 2. Normalizar para que la suma de las proporciones sea 1 (si la suma no es cero)
    sum_proportions = raw_proportions.sum()
    if sum_proportions > 0:
        normalized_proportions = raw_proportions / sum_proportions
    else:
        normalized_proportions = raw_proportions # Se queda como un vector de ceros
        
    all_proportions.append(normalized_proportions.flatten())

# Convertimos la lista de resultados en un DataFrame
deconvolution_results_df = pd.DataFrame(
    all_proportions,
    index=bulk_linear_df.index,
    columns=cell_types
)

print("\n--- Deconvolución completada ---")
print("Dimensiones del DataFrame de resultados:", deconvolution_results_df.shape)
display(deconvolution_results_df.head())


## 3.3. Verificación de los Resultados

Verificamos que las proporciones estimadas para cada muestra sumen 1.

In [ ]:
# Calculamos la suma de las proporciones para cada muestra
row_sums = deconvolution_results_df.sum(axis=1)

# Usamos np.allclose para verificar si todas las sumas son aproximadamente 1
try:
    assert np.allclose(row_sums, 1.0)
    print("\n[OK] Verificación exitosa: Todas las proporciones por muestra suman 1.")
except AssertionError:
    print("\n[ERROR] ¡Las proporciones no suman 1! Revisa el paso de normalización.")
    display(row_sums.describe())